# 05 — Evaluation & Analysis

Compare MTL vs baselines, ablation studies, feature importance, and representation visualization.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.data.dataset import AsteroidDataset
from src.models.mtl_model import AsteroidMTLModel
from src.models.losses import MultiTaskLoss
from src.training.trainer import Trainer
from src.utils.visualization import plot_confusion_matrix

sns.set_theme(style='whitegrid')
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Load Best Model & Test Data

In [ ]:
test_ds = AsteroidDataset('../data/processed/test.parquet')
test_loader = DataLoader(test_ds, batch_size=2048, shuffle=False)

label_encoder = joblib.load('../data/processed/label_encoder.joblib')
n_classes = len(label_encoder.classes_)

model = AsteroidMTLModel(
    n_features=test_ds.n_features,
    n_classes=n_classes,
    backbone_layers=[512, 256, 128, 64],
    backbone_dropouts=[0.3, 0.3, 0.2, 0.2],
    head_hidden=32,
    head_dropout=0.1,
)

checkpoint = torch.load('../models/mtl_model.pt', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()
print('Model loaded.')

## 2. Extract Predictions and Shared Representations

In [ ]:
all_repr, all_class_true, all_class_pred = [], [], []
all_h_true, all_h_pred, all_h_mask = [], [], []
all_d_true, all_d_pred, all_d_mask = [], [], []

with torch.no_grad():
    for batch in test_loader:
        features = batch['features'].to(device)
        outputs = model(features)
        
        all_repr.append(outputs['shared_repr'].cpu().numpy())
        all_class_true.append(batch['class_label'].numpy())
        all_class_pred.append(outputs['class_logits'].argmax(dim=1).cpu().numpy())
        all_h_true.append(batch['h_target'].numpy())
        all_h_pred.append(outputs['h_pred'].cpu().numpy())
        all_h_mask.append(batch['h_mask'].numpy())
        all_d_true.append(batch['diameter_target'].numpy())
        all_d_pred.append(outputs['diameter_pred'].cpu().numpy())
        all_d_mask.append(batch['diameter_mask'].numpy())

representations = np.concatenate(all_repr)
y_cls_true = np.concatenate(all_class_true)
y_cls_pred = np.concatenate(all_class_pred)
y_h_true = np.concatenate(all_h_true)
y_h_pred = np.concatenate(all_h_pred)
h_mask = np.concatenate(all_h_mask).astype(bool)
y_d_true = np.concatenate(all_d_true)
y_d_pred = np.concatenate(all_d_pred)
d_mask = np.concatenate(all_d_mask).astype(bool)

print(f'Test samples: {len(representations)}')
print(f'Shared representation dim: {representations.shape[1]}')

## 3. Confusion Matrix

In [ ]:
fig = plot_confusion_matrix(y_cls_true, y_cls_pred, label_encoder.classes_.tolist())
plt.show()

from sklearn.metrics import classification_report
print(classification_report(y_cls_true, y_cls_pred, target_names=label_encoder.classes_))

## 4. Regression Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# H magnitude
ax = axes[0]
ax.scatter(y_h_true[h_mask], y_h_pred[h_mask], alpha=0.1, s=1)
lims = [y_h_true[h_mask].min(), y_h_true[h_mask].max()]
ax.plot(lims, lims, 'r--', alpha=0.8)
ax.set_xlabel('True H')
ax.set_ylabel('Predicted H')
ax.set_title('H Magnitude: True vs Predicted')

# Diameter
ax = axes[1]
ax.scatter(y_d_true[d_mask], y_d_pred[d_mask], alpha=0.1, s=1)
lims = [y_d_true[d_mask].min(), y_d_true[d_mask].max()]
ax.plot(lims, lims, 'r--', alpha=0.8)
ax.set_xlabel('True log(1+diameter)')
ax.set_ylabel('Predicted log(1+diameter)')
ax.set_title('Diameter: True vs Predicted')

plt.tight_layout()
plt.show()

## 5. t-SNE of Shared Representations

In [ ]:
# Sample for t-SNE (full dataset is too large)
n_sample = 10000
rng = np.random.RandomState(42)
idx = rng.choice(len(representations), n_sample, replace=False)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
repr_2d = tsne.fit_transform(representations[idx])

fig, ax = plt.subplots(figsize=(12, 10))
classes = y_cls_true[idx]
for c in range(n_classes):
    mask = classes == c
    ax.scatter(repr_2d[mask, 0], repr_2d[mask, 1], 
               alpha=0.4, s=5, label=label_encoder.classes_[c])
ax.legend(markerscale=5, fontsize=9)
ax.set_title('t-SNE of Shared Backbone Representations')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

## 6. Comparison Table: MTL vs Baselines

Fill in baseline results from notebook 03 and MTL results from notebook 04.

In [ ]:
from src.training.metrics import classification_metrics, regression_metrics

mtl_cls = classification_metrics(y_cls_true, y_cls_pred)
mtl_h = regression_metrics(y_h_true, y_h_pred, h_mask.astype(float))
mtl_d = regression_metrics(y_d_true, y_d_pred, d_mask.astype(float))

print('=== MTL Model Test Results ===')
print(f'\nClassification:')
print(f'  Accuracy:    {mtl_cls["accuracy"]:.4f}')
print(f'  F1 (macro):  {mtl_cls["f1_macro"]:.4f}')
print(f'  F1 (weight): {mtl_cls["f1_weighted"]:.4f}')
print(f'\nH Magnitude:')
print(f'  MAE:  {mtl_h["mae"]:.4f}')
print(f'  RMSE: {mtl_h["rmse"]:.4f}')
print(f'  R²:   {mtl_h["r2"]:.4f}')
print(f'\nDiameter (log-scale):')
print(f'  MAE:  {mtl_d["mae"]:.4f}')
print(f'  RMSE: {mtl_d["rmse"]:.4f}')
print(f'  R²:   {mtl_d["r2"]:.4f}')

print('\n\n--- Copy baseline results from notebook 03 to complete the comparison table ---')